# signal-enrichment-agent — Exploratory Analysis

This notebook explores the **Gold layer** produced by the enrichment pipeline.  
Use it to validate enrichment quality, spot biases, and build intuition before promoting to production.

**Sections**
1. Setup & connection  
2. Dataset overview  
3. Sentiment distribution  
4. Category distribution  
5. Entity frequency analysis  
6. Cost & token usage analysis  
7. Quality checks — low-confidence records  
8. Export summary report


## 1. Setup & Connection

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── styling ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (10, 4)})

# ── connection ────────────────────────────────────────────────────────────
DB_PATH = Path("../data/warehouse.duckdb")   # adjust if needed
GOLD_TABLE = "gold.reviews"
COST_LOG   = Path("../logs/cost.jsonl")

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected to: {DB_PATH.resolve()}")
print(f"DuckDB version: {duckdb.__version__}")


## 2. Dataset Overview

In [ ]:
df = con.execute(f"SELECT * FROM {GOLD_TABLE}").fetchdf()

print(f"Total enriched records : {len(df):,}")
print(f"Columns                : {list(df.columns)}")
print(f"Date range             : {df['date'].min()} → {df['date'].max()}")
print(f"Enrichment errors      : {df['enrichment_error'].sum()} ({df['enrichment_error'].mean()*100:.1f}%)")
print()
df.head(3)


In [ ]:
# Null rates per enrichment field
enrichment_cols = ["sentiment", "sentiment_confidence", "category", "subcategory",
                   "entities_brands", "entities_locations", "entities_persons"]
null_rates = df[enrichment_cols].isnull().mean().rename("null_rate").to_frame()
null_rates["null_rate"] = null_rates["null_rate"].map("{:.1%}".format)
null_rates


## 3. Sentiment Distribution

In [ ]:
sentiment_counts = df["sentiment"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = {"positive": "#4CAF50", "neutral": "#90A4AE", "negative": "#EF5350"}
bars = axes[0].bar(
    sentiment_counts.index,
    sentiment_counts.values,
    color=[colors.get(s, "#90A4AE") for s in sentiment_counts.index],
    width=0.5, edgecolor="white", linewidth=1.5,
)
axes[0].bar_label(bars, fmt="{:,.0f}", padding=4)
axes[0].set_title("Record count by sentiment", fontweight="bold")
axes[0].set_ylabel("Records")
axes[0].set_xlabel("")

# Confidence distribution by sentiment
df_valid = df.dropna(subset=["sentiment", "sentiment_confidence"])
for sentiment, grp in df_valid.groupby("sentiment"):
    axes[1].hist(
        grp["sentiment_confidence"], bins=20, alpha=0.6,
        label=sentiment, color=colors.get(sentiment, "#90A4AE"),
    )
axes[1].set_title("Confidence score distribution by sentiment", fontweight="bold")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nMedian confidence per sentiment:")
print(df_valid.groupby("sentiment")["sentiment_confidence"].median().round(3))


## 4. Category Distribution

In [ ]:
cat_counts = df["category"].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Horizontal bar — top categories
axes[0].barh(cat_counts.index[::-1], cat_counts.values[::-1],
             color=sns.color_palette("muted", len(cat_counts)))
axes[0].set_title("Top categories", fontweight="bold")
axes[0].set_xlabel("Records")
for i, (val, label) in enumerate(zip(cat_counts.values[::-1], cat_counts.index[::-1])):
    axes[0].text(val + 0.5, i, str(val), va="center", fontsize=9)

# Sentiment breakdown within each category (top 8)
top_cats = cat_counts.head(8).index.tolist()
pivot = (
    df[df["category"].isin(top_cats)]
    .groupby(["category", "sentiment"])
    .size()
    .unstack(fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[["positive", "neutral", "negative"]].plot(
    kind="barh", ax=axes[1], stacked=True,
    color=["#4CAF50", "#90A4AE", "#EF5350"], edgecolor="white",
)
axes[1].set_title("Sentiment mix per category (%)", fontweight="bold")
axes[1].set_xlabel("Percentage")
axes[1].legend(loc="lower right")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()


In [ ]:
# Subcategory breakdown for the most common category
top_category = cat_counts.index[0]
sub_counts = (
    df[df["category"] == top_category]["subcategory"]
    .value_counts()
    .head(10)
)
print(f"Top subcategories within '{top_category}':")
print(sub_counts.to_string())


## 5. Entity Frequency Analysis

In [ ]:
def explode_json_list(series: pd.Series) -> pd.Series:
    """Flatten a column of JSON arrays into a flat Series of values."""
    exploded = []
    for cell in series.dropna():
        items = json.loads(cell) if isinstance(cell, str) else cell
        if isinstance(items, list):
            exploded.extend(items)
    return pd.Series(exploded)

brands    = explode_json_list(df["entities_brands"])
locations = explode_json_list(df["entities_locations"])
persons   = explode_json_list(df["entities_persons"])

print(f"Unique brands   : {brands.nunique()}")
print(f"Unique locations: {locations.nunique()}")
print(f"Unique persons  : {persons.nunique()}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, series, title, color in [
    (axes[0], brands,    "Top Brands",     "#5C6BC0"),
    (axes[1], locations, "Top Locations",  "#26A69A"),
    (axes[2], persons,   "Top Persons",    "#FFA726"),
]:
    counts = series.value_counts().head(10)
    if counts.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.barh(counts.index[::-1], counts.values[::-1], color=color, alpha=0.85)
        for i, val in enumerate(counts.values[::-1]):
            ax.text(val + 0.1, i, str(val), va="center", fontsize=8)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Mentions")

plt.tight_layout()
plt.show()


## 6. Cost & Token Usage Analysis

In [ ]:
if COST_LOG.exists():
    cost_df = pd.read_json(COST_LOG, lines=True)
    cost_df["timestamp"] = pd.to_datetime(cost_df["timestamp"])

    print("=== Cost Summary ===")
    print(f"Total calls      : {len(cost_df):,}")
    print(f"Total tokens     : {cost_df['total_tokens'].sum():,}")
    print(f"Total cost (USD) : ${cost_df['cost_usd'].sum():.4f}")
    print()

    by_agent = cost_df.groupby("agent").agg(
        calls=("cost_usd", "count"),
        total_tokens=("total_tokens", "sum"),
        total_cost=("cost_usd", "sum"),
        avg_tokens=("total_tokens", "mean"),
    ).round(4)
    print("=== Breakdown by Agent ===")
    print(by_agent.to_string())
else:
    print(f"Cost log not found at {COST_LOG}. Run the pipeline first.")
    cost_df = pd.DataFrame()


In [ ]:
if not cost_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Cumulative cost over time
    cost_df_sorted = cost_df.sort_values("timestamp")
    cost_df_sorted["cumulative_cost"] = cost_df_sorted["cost_usd"].cumsum()
    axes[0].plot(cost_df_sorted["timestamp"], cost_df_sorted["cumulative_cost"],
                 color="#5C6BC0", linewidth=2)
    axes[0].fill_between(cost_df_sorted["timestamp"], cost_df_sorted["cumulative_cost"],
                         alpha=0.15, color="#5C6BC0")
    axes[0].set_title("Cumulative cost over time (USD)", fontweight="bold")
    axes[0].set_ylabel("USD")
    axes[0].xaxis.set_tick_params(rotation=30)

    # Token distribution per agent
    cost_df.boxplot(column="total_tokens", by="agent", ax=axes[1], 
                    patch_artist=True, 
                    boxprops=dict(facecolor="#E8EAF6"))
    axes[1].set_title("Token distribution per agent", fontweight="bold")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Tokens per call")
    plt.suptitle("")

    plt.tight_layout()
    plt.show()


## 7. Quality Checks — Low-Confidence Records

In [ ]:
CONFIDENCE_THRESHOLD = 0.70

low_conf = df[
    df["sentiment_confidence"].notna() &
    (df["sentiment_confidence"] < CONFIDENCE_THRESHOLD)
].sort_values("sentiment_confidence")

print(f"Records below {CONFIDENCE_THRESHOLD} confidence threshold: {len(low_conf)} "
      f"({len(low_conf)/len(df)*100:.1f}%)")
print()

if not low_conf.empty:
    display_cols = ["review_id", "review_text", "sentiment", "sentiment_confidence"]
    print(low_conf[display_cols].head(10).to_string(index=False))


In [ ]:
# Records with enrichment errors
errors = df[df["enrichment_error"] == True]
print(f"Records with enrichment errors: {len(errors)}")
if not errors.empty:
    print("\nSample error records:")
    print(errors[["review_id", "review_text"]].head(5).to_string(index=False))


## 8. Export Summary Report

In [ ]:
summary = {
    "total_records": int(len(df)),
    "enrichment_error_rate": float(df["enrichment_error"].mean()),
    "sentiment_distribution": df["sentiment"].value_counts(normalize=True).round(4).to_dict(),
    "avg_sentiment_confidence": float(df["sentiment_confidence"].mean()),
    "top_categories": df["category"].value_counts().head(5).to_dict(),
    "unique_brands": int(brands.nunique()),
    "unique_locations": int(locations.nunique()),
}

import json
output_path = Path("../data/enrichment_summary.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

print("Summary written to:", output_path.resolve())
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
con.close()
print("Connection closed.")
